# Advanced SQL for Data Engineering
## Beyond Foundations — Production-Level SQL Patterns

**What you'll learn:**
- Advanced window functions (frames, NTILE, PERCENT_RANK, cumulative distributions)
- Recursive CTEs for hierarchical data
- PIVOT and UNPIVOT for reshaping data
- Semi-joins, anti-joins, and lateral joins
- Query optimization (EXPLAIN, predicate pushdown, partition pruning)
- Cohort analysis and retention calculations
- Funnel analytics and conversion tracking
- Session detection and gap analysis
- JSON/semi-structured data in SQL
- Advanced MERGE patterns and CDC
- 20+ interview-level challenges

**Prerequisites:** 02_sql_foundation.ipynb (must have techmart_dw database created)
**Estimated Time:** 8-10 hours

---

> This notebook assumes you are comfortable with basic SQL (SELECT, JOIN,
> GROUP BY, simple window functions). We go DEEP into the patterns that
> separate junior from senior data engineers.

---

---
# Section 1: Advanced Window Functions

## Beyond ROW_NUMBER -- The Full Window Function Toolkit

The foundation notebook covered ROW_NUMBER, RANK, LAG/LEAD.
Now we go deeper: custom frames, percentiles, cumulative distributions,
and multi-window queries.

## Window Frame Specification (the part most people skip)

```sql
function() OVER (
    PARTITION BY ...
    ORDER BY ...
    ROWS BETWEEN <start> AND <end>  -- Physical rows
    -- or --
    RANGE BETWEEN <start> AND <end> -- Logical values
)

-- Frame boundaries:
-- UNBOUNDED PRECEDING = first row of partition
-- n PRECEDING = n rows/values before current
-- CURRENT ROW = the current row
-- n FOLLOWING = n rows/values after current
-- UNBOUNDED FOLLOWING = last row of partition
```

In [ ]:
%sql
USE techmart_dw;

-- Advanced Window: Multiple windows in one query
-- Customer order analytics with running metrics
SELECT
    customer_id,
    order_id,
    order_date,
    total_amount,
    -- Running total per customer
    SUM(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total,
    -- 3-order moving average
    ROUND(AVG(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3,
    -- Difference from previous order
    total_amount - LAG(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
    ) AS diff_from_prev,
    -- Percentage of customer's total spending
    ROUND(total_amount * 100.0 / SUM(total_amount) OVER (
        PARTITION BY customer_id
    ), 2) AS pct_of_total
FROM orders
WHERE customer_id IN (1, 2, 3) AND status = 'completed'
ORDER BY customer_id, order_date
LIMIT 30;

In [ ]:
%sql
-- PERCENT_RANK and CUME_DIST: percentile calculations
-- Find which percentile each customer falls in by lifetime value
SELECT
    customer_id,
    first_name,
    customer_segment,
    lifetime_value,
    ROUND(PERCENT_RANK() OVER (ORDER BY lifetime_value), 4) AS percentile_rank,
    ROUND(CUME_DIST() OVER (ORDER BY lifetime_value), 4) AS cumulative_dist,
    NTILE(10) OVER (ORDER BY lifetime_value) AS decile,
    NTILE(100) OVER (ORDER BY lifetime_value) AS percentile_bucket
FROM customers
WHERE is_active = true
ORDER BY lifetime_value DESC
LIMIT 20;

In [ ]:
%sql
-- FIRST_VALUE / LAST_VALUE / NTH_VALUE
-- Compare each order to the first and best order per customer
SELECT
    customer_id,
    order_id,
    order_date,
    total_amount,
    FIRST_VALUE(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS first_order_amount,
    LAST_VALUE(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS last_order_amount,
    MAX(total_amount) OVER (PARTITION BY customer_id) AS max_order_amount,
    total_amount - FIRST_VALUE(total_amount) OVER (
        PARTITION BY customer_id ORDER BY order_date
    ) AS growth_from_first
FROM orders
WHERE customer_id BETWEEN 1 AND 5 AND status = 'completed'
ORDER BY customer_id, order_date;

In [ ]:
%sql
-- ============================================
-- YOUR TURN: Window Frame Mastery
-- ============================================
-- For each day, calculate:
-- 1. Daily revenue
-- 2. 7-day trailing moving average (current + 6 preceding)
-- 3. Month-to-date cumulative revenue (reset each month)
-- 4. Percentage of monthly total (how much of this month's revenue came today)
-- 5. Day-over-day revenue change
--
-- Filter to completed orders, show last 30 days
--
-- Write your query below:


In [ ]:
%sql
-- SOLUTION: Daily Revenue with Advanced Windows
WITH daily_revenue AS (
    SELECT
        order_date,
        DATE_TRUNC('month', order_date) AS month_start,
        SUM(total_amount) AS revenue,
        COUNT(*) AS order_count
    FROM orders
    WHERE status = 'completed'
    GROUP BY order_date, DATE_TRUNC('month', order_date)
)
SELECT
    order_date,
    ROUND(revenue, 2) AS daily_revenue,
    -- 7-day moving average
    ROUND(AVG(revenue) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS ma_7d,
    -- Month-to-date (partition resets each month)
    ROUND(SUM(revenue) OVER (
        PARTITION BY month_start
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS mtd_revenue,
    -- Pct of monthly total
    ROUND(revenue * 100.0 / SUM(revenue) OVER (PARTITION BY month_start), 2) AS pct_of_month,
    -- Day-over-day change
    ROUND(revenue - LAG(revenue) OVER (ORDER BY order_date), 2) AS dod_change
FROM daily_revenue
ORDER BY order_date DESC
LIMIT 30;

---
# Section 2: Advanced Aggregation Patterns

## GROUPING SETS, ROLLUP, and CUBE

These generate multiple levels of aggregation in a single query --
equivalent to running multiple GROUP BY queries and UNION ALL-ing them.

- **GROUPING SETS**: Specify exact combinations you want
- **ROLLUP**: Hierarchical subtotals (year > month > day)
- **CUBE**: All possible combinations (2^n groups)

In [ ]:
%sql
-- ROLLUP: Hierarchical subtotals
-- Revenue by year > month with subtotals and grand total
SELECT
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    payment_method,
    COUNT(*) AS order_count,
    ROUND(SUM(total_amount), 2) AS revenue,
    GROUPING_ID(YEAR(order_date), MONTH(order_date), payment_method) AS grouping_level
FROM orders
WHERE status = 'completed' AND order_date >= '2023-01-01'
GROUP BY ROLLUP(YEAR(order_date), MONTH(order_date), payment_method)
HAVING YEAR(order_date) IS NOT NULL OR GROUPING_ID(YEAR(order_date), MONTH(order_date), payment_method) = 7
ORDER BY order_year, order_month, payment_method
LIMIT 40;

In [ ]:
%sql
-- CUBE: All dimension combinations
-- Good for building summary tables / OLAP cubes
SELECT
    COALESCE(CAST(YEAR(order_date) AS STRING), 'ALL') AS year,
    COALESCE(order_source, 'ALL') AS source,
    COALESCE(status, 'ALL') AS status,
    COUNT(*) AS orders,
    ROUND(SUM(total_amount), 2) AS revenue
FROM orders
WHERE order_date >= '2023-06-01'
GROUP BY CUBE(YEAR(order_date), order_source, status)
HAVING COUNT(*) > 100
ORDER BY year, source, status
LIMIT 30;

---
# Section 3: Cohort Analysis & Retention

## The Most Asked Analytics Pattern in DE Interviews

Cohort analysis groups users by when they first appeared (acquisition cohort)
and tracks their behavior over time. This is THE bread-and-butter query for
product analytics, marketing teams, and investor reporting.

```
Example output (retention matrix):
Cohort    Month 0   Month 1   Month 2   Month 3
2023-01   1000      450       320       280
2023-02   1200      510       350       290
2023-03   800       380       250       ...
```

In [ ]:
%sql
-- Cohort Retention Analysis
-- Step 1: Find each customer's cohort (first order month)
-- Step 2: Calculate months since first order for each subsequent order
-- Step 3: Count retained customers at each interval

WITH customer_cohort AS (
    -- Each customer's first order month = their cohort
    SELECT
        customer_id,
        DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
customer_activity AS (
    -- Each customer's activity months
    SELECT DISTINCT
        o.customer_id,
        cc.cohort_month,
        DATE_TRUNC('month', o.order_date) AS activity_month,
        MONTHS_BETWEEN(DATE_TRUNC('month', o.order_date), cc.cohort_month) AS months_since_first
    FROM orders o
    JOIN customer_cohort cc ON o.customer_id = cc.customer_id
    WHERE o.status = 'completed'
)
SELECT
    cohort_month,
    COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END) AS month_0,
    COUNT(DISTINCT CASE WHEN months_since_first = 1 THEN customer_id END) AS month_1,
    COUNT(DISTINCT CASE WHEN months_since_first = 2 THEN customer_id END) AS month_2,
    COUNT(DISTINCT CASE WHEN months_since_first = 3 THEN customer_id END) AS month_3,
    COUNT(DISTINCT CASE WHEN months_since_first = 6 THEN customer_id END) AS month_6,
    -- Retention rates
    ROUND(COUNT(DISTINCT CASE WHEN months_since_first = 1 THEN customer_id END) * 100.0 /
          NULLIF(COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END), 0), 1) AS retention_m1_pct
FROM customer_activity
WHERE cohort_month >= '2022-01-01'
GROUP BY cohort_month
ORDER BY cohort_month
LIMIT 20;

In [ ]:
%sql
-- ============================================
-- YOUR TURN: Customer Lifetime Value by Cohort
-- ============================================
-- Calculate the AVERAGE cumulative revenue per customer at each month
-- since their first order, grouped by cohort.
-- This shows how much revenue each cohort generates over time.
--
-- Output: cohort_month, months_since_first, avg_cumulative_revenue, customers_active
-- Filter to cohorts from 2022+
--
-- Write your query below:


In [ ]:
%sql
-- SOLUTION: Cumulative LTV by Cohort
WITH customer_cohort AS (
    SELECT customer_id, DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders WHERE status = 'completed' GROUP BY customer_id
),
monthly_spend AS (
    SELECT
        o.customer_id,
        cc.cohort_month,
        CAST(MONTHS_BETWEEN(DATE_TRUNC('month', o.order_date), cc.cohort_month) AS INT) AS months_since_first,
        SUM(o.total_amount) AS monthly_revenue
    FROM orders o
    JOIN customer_cohort cc ON o.customer_id = cc.customer_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id, cc.cohort_month,
             CAST(MONTHS_BETWEEN(DATE_TRUNC('month', o.order_date), cc.cohort_month) AS INT)
),
cumulative AS (
    SELECT
        customer_id,
        cohort_month,
        months_since_first,
        SUM(monthly_revenue) OVER (
            PARTITION BY customer_id ORDER BY months_since_first
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue
    FROM monthly_spend
)
SELECT
    cohort_month,
    months_since_first,
    COUNT(DISTINCT customer_id) AS customers_active,
    ROUND(AVG(cumulative_revenue), 2) AS avg_cumulative_ltv
FROM cumulative
WHERE cohort_month >= '2022-01-01' AND months_since_first <= 12
GROUP BY cohort_month, months_since_first
ORDER BY cohort_month, months_since_first
LIMIT 50;

---
# Section 4: Sessionization & Gap Detection

## Detecting Sessions in Event Streams

A session is a group of events from the same user with no gap larger than
a threshold (e.g., 30 minutes). This is fundamental for:
- Web analytics (user sessions)
- IoT data (device activity windows)
- Fraud detection (transaction bursts)

## The Pattern: Detect gaps -> assign session IDs

In [ ]:
%sql
-- Sessionization: Group orders into "shopping sessions"
-- A session breaks if > 30 days between orders from same customer
WITH order_gaps AS (
    SELECT
        customer_id,
        order_id,
        order_date,
        total_amount,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date,
        DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_gap
    FROM orders
    WHERE status = 'completed'
),
session_starts AS (
    SELECT
        *,
        CASE WHEN days_gap IS NULL OR days_gap > 30 THEN 1 ELSE 0 END AS is_new_session
    FROM order_gaps
),
sessions AS (
    SELECT
        *,
        SUM(is_new_session) OVER (PARTITION BY customer_id ORDER BY order_date) AS session_id
    FROM session_starts
)
SELECT
    customer_id,
    session_id,
    COUNT(*) AS orders_in_session,
    MIN(order_date) AS session_start,
    MAX(order_date) AS session_end,
    DATEDIFF(MAX(order_date), MIN(order_date)) AS session_duration_days,
    ROUND(SUM(total_amount), 2) AS session_revenue
FROM sessions
WHERE customer_id BETWEEN 1 AND 10
GROUP BY customer_id, session_id
ORDER BY customer_id, session_start
LIMIT 30;

In [ ]:
%sql
-- Gap Detection: Find gaps in sequential data
-- Useful for: missing dates, skipped IDs, inventory gaps
WITH date_spine AS (
    SELECT DATE_ADD('2023-01-01', id) AS expected_date
    FROM (SELECT explode(sequence(0, 364)) AS id)
),
actual_dates AS (
    SELECT DISTINCT order_date
    FROM orders
    WHERE order_date BETWEEN '2023-01-01' AND '2023-12-31'
)
SELECT
    ds.expected_date AS missing_date,
    DAYOFWEEK(ds.expected_date) AS day_of_week
FROM date_spine ds
LEFT JOIN actual_dates ad ON ds.expected_date = ad.order_date
WHERE ad.order_date IS NULL
ORDER BY ds.expected_date
LIMIT 20;

---
# Section 5: Data Reshaping & Semi-Structured Data

## PIVOT: Rows to Columns
## UNPIVOT: Columns to Rows
## JSON Functions: Parsing nested data in SQL

In [ ]:
%sql
-- PIVOT: Monthly revenue by payment method (rows -> columns)
SELECT *
FROM (
    SELECT
        date_format(order_date, 'yyyy-MM') AS month,
        payment_method,
        total_amount
    FROM orders
    WHERE status = 'completed' AND order_date >= '2023-07-01'
)
PIVOT (
    ROUND(SUM(total_amount), 2) AS revenue
    FOR payment_method IN ('credit_card', 'debit_card', 'paypal', 'bank_transfer', 'crypto')
)
ORDER BY month
LIMIT 12;

In [ ]:
%sql
-- JSON parsing: Extract fields from metadata column
SELECT
    customer_id,
    first_name,
    metadata,
    GET_JSON_OBJECT(metadata, '$.source') AS acquisition_source,
    GET_JSON_OBJECT(metadata, '$.loyalty_tier') AS loyalty_tier,
    CASE
        WHEN CAST(GET_JSON_OBJECT(metadata, '$.loyalty_tier') AS INT) >= 4 THEN 'VIP'
        WHEN CAST(GET_JSON_OBJECT(metadata, '$.loyalty_tier') AS INT) >= 2 THEN 'Regular'
        ELSE 'New'
    END AS tier_category
FROM customers
WHERE metadata IS NOT NULL
LIMIT 15;

In [ ]:
%sql
-- JSON: Parse payment gateway responses
SELECT
    payment_id,
    order_id,
    amount,
    payment_status,
    gateway_response,
    GET_JSON_OBJECT(gateway_response, '$.gateway') AS gateway,
    GET_JSON_OBJECT(gateway_response, '$.response_code') AS response_code,
    GET_JSON_OBJECT(gateway_response, '$.auth_code') AS auth_code,
    CASE GET_JSON_OBJECT(gateway_response, '$.response_code')
        WHEN '200' THEN 'Success'
        WHEN '201' THEN 'Created'
        WHEN '400' THEN 'Bad Request'
        ELSE 'Unknown'
    END AS response_meaning
FROM payments
LIMIT 15;

---
# Section 6: Advanced MERGE & CDC Patterns

## Beyond Basic MERGE: Multi-Action, Conditional, Type 2 SCD

MERGE is THE workhorse for data engineering ETL. Here we cover patterns
that handle real-world complexity: conditional updates, soft deletes,
SCD Type 2 with effective dating, and deduplication during merge.

In [ ]:
%sql
-- Create a staging table to simulate incoming data
CREATE OR REPLACE TEMP VIEW customers_staging AS
SELECT * FROM (VALUES
    (1, 'UpdatedFirst1', 'Last1', 'updated1@gmail.com', 'Premium', 99999.99),
    (5001, 'BrandNew', 'Customer', 'brandnew@gmail.com', 'New', 100.00),
    (5002, 'Another', 'NewOne', 'another@gmail.com', 'Basic', 250.00)
) AS t(customer_id, first_name, last_name, email, customer_segment, lifetime_value);

-- Show what will happen (dry run)
SELECT
    COALESCE(s.customer_id, t.customer_id) AS customer_id,
    CASE
        WHEN t.customer_id IS NULL THEN 'INSERT'
        WHEN s.customer_id IS NULL THEN 'DELETE (not in source)'
        ELSE 'UPDATE'
    END AS action,
    t.first_name AS current_name,
    s.first_name AS incoming_name
FROM customers_staging s
FULL OUTER JOIN customers t ON s.customer_id = t.customer_id
WHERE s.customer_id IS NOT NULL;

In [ ]:
%sql
-- Advanced MERGE with conditional logic
-- Only update if the incoming data is actually different (avoid unnecessary writes)
MERGE INTO customers AS target
USING customers_staging AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND (
    target.first_name != source.first_name OR
    target.email != source.email OR
    target.customer_segment != source.customer_segment
) THEN UPDATE SET
    first_name = source.first_name,
    last_name = source.last_name,
    email = source.email,
    customer_segment = source.customer_segment,
    lifetime_value = source.lifetime_value,
    updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    customer_id, first_name, last_name, email, customer_segment,
    lifetime_value, is_active, created_at, updated_at
) VALUES (
    source.customer_id, source.first_name, source.last_name, source.email,
    source.customer_segment, source.lifetime_value, true, current_timestamp(), current_timestamp()
);

---
# Section 7: Query Optimization

## Reading Execution Plans & Making Queries Fast

Understanding EXPLAIN output is what separates data engineers who
write queries from those who write EFFICIENT queries.

Key things to look for:
- **Scan type**: Full table scan vs partition pruning vs index scan
- **Shuffle/Exchange**: Data movement between nodes (expensive)
- **Join strategy**: Broadcast vs Sort-Merge vs Shuffle Hash
- **Predicate pushdown**: Filters pushed to storage layer

In [ ]:
%sql
-- Compare two approaches: subquery vs window function
-- Approach 1: Find customers whose latest order is above average
EXPLAIN FORMATTED
SELECT customer_id, order_date, total_amount
FROM orders o
WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
  AND order_date = (SELECT MAX(order_date) FROM orders o2 WHERE o2.customer_id = o.customer_id);

In [ ]:
%sql
-- Approach 2: Same logic with window functions (typically faster)
EXPLAIN FORMATTED
WITH ranked AS (
    SELECT
        customer_id, order_date, total_amount,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn,
        AVG(total_amount) OVER () AS overall_avg
    FROM orders
)
SELECT customer_id, order_date, total_amount
FROM ranked
WHERE rn = 1 AND total_amount > overall_avg;

In [ ]:
%sql
-- Optimization: Partition pruning demonstration
-- Without partition filter (full scan):
EXPLAIN FORMATTED
SELECT COUNT(*), SUM(total_amount)
FROM orders
WHERE status = 'completed';

In [ ]:
%sql
-- With partition filter (pruned scan):
EXPLAIN FORMATTED
SELECT COUNT(*), SUM(total_amount)
FROM orders
WHERE status = 'completed' AND order_date >= '2024-01-01';

---
# Section 8: Interview-Level SQL Challenges

## Real problems from FAANG / Databricks / Uber / Airbnb interviews

These are the types of questions asked in SQL rounds for DE positions.
Time yourself: 15-20 minutes per problem.

In [ ]:
%sql
-- ============================================
-- CHALLENGE 1: Consecutive Days (Gap & Island)
-- ============================================
-- Find customers who ordered on 3+ CONSECUTIVE days.
-- This is the classic "gaps and islands" problem.
--
-- Expected output: customer_id, streak_start, streak_end, streak_length

WITH daily_orders AS (
    SELECT DISTINCT customer_id, order_date
    FROM orders WHERE status = 'completed'
),
with_groups AS (
    SELECT
        customer_id,
        order_date,
        -- Subtract row number from date: consecutive dates get same group
        DATE_SUB(order_date, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date)) AS grp
    FROM daily_orders
)
SELECT
    customer_id,
    MIN(order_date) AS streak_start,
    MAX(order_date) AS streak_end,
    COUNT(*) AS streak_length
FROM with_groups
GROUP BY customer_id, grp
HAVING COUNT(*) >= 3
ORDER BY streak_length DESC, customer_id
LIMIT 20;

In [ ]:
%sql
-- ============================================
-- CHALLENGE 2: Year-over-Year Growth
-- ============================================
-- For each month, calculate:
-- 1. Current month revenue
-- 2. Same month last year revenue
-- 3. YoY growth percentage
-- 4. Flag months with > 20% decline as 'ALERT'

WITH monthly AS (
    SELECT
        YEAR(order_date) AS yr,
        MONTH(order_date) AS mo,
        date_format(order_date, 'yyyy-MM') AS month,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY YEAR(order_date), MONTH(order_date), date_format(order_date, 'yyyy-MM')
)
SELECT
    curr.month,
    curr.revenue AS current_revenue,
    prev.revenue AS prior_year_revenue,
    ROUND((curr.revenue - prev.revenue) / NULLIF(prev.revenue, 0) * 100, 2) AS yoy_growth_pct,
    CASE
        WHEN prev.revenue IS NULL THEN 'No Prior Data'
        WHEN (curr.revenue - prev.revenue) / prev.revenue < -0.20 THEN 'ALERT: >20% Decline'
        WHEN (curr.revenue - prev.revenue) / prev.revenue > 0.20 THEN 'Strong Growth'
        ELSE 'Normal'
    END AS status
FROM monthly curr
LEFT JOIN monthly prev ON curr.yr = prev.yr + 1 AND curr.mo = prev.mo
ORDER BY curr.month DESC
LIMIT 24;

In [ ]:
%sql
-- ============================================
-- CHALLENGE 3: Running Top-N per Group
-- ============================================
-- For each product category, find the top 3 products by total revenue.
-- Show category, product_name, revenue, and rank.

WITH product_revenue AS (
    SELECT
        p.category,
        p.product_name,
        ROUND(SUM(oi.line_total), 2) AS total_revenue,
        COUNT(DISTINCT oi.order_id) AS orders_count
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
    FROM product_revenue
)
SELECT category, product_name, total_revenue, orders_count, rank_in_category
FROM ranked
WHERE rank_in_category <= 3
ORDER BY category, rank_in_category;

In [ ]:
%sql
-- ============================================
-- YOUR TURN: Challenge 4 -- Customer Churn Detection
-- ============================================
-- A customer is "churned" if they have not ordered in the last 90 days
-- but DID order before that.
--
-- Find all churned customers with:
-- 1. customer_id, first_name
-- 2. last_order_date
-- 3. days_since_last_order
-- 4. total_orders (lifetime)
-- 5. total_spent (lifetime)
-- 6. customer_segment
--
-- Sort by total_spent DESC (we want to recover high-value churners first)
-- Show top 20
--
-- Write your query below:


In [ ]:
%sql
-- SOLUTION: Customer Churn Detection
WITH customer_last_order AS (
    SELECT
        customer_id,
        MAX(order_date) AS last_order_date,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS total_spent
    FROM orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.first_name,
    c.customer_segment,
    clo.last_order_date,
    DATEDIFF(CURRENT_DATE(), clo.last_order_date) AS days_since_last_order,
    clo.total_orders,
    clo.total_spent
FROM customer_last_order clo
JOIN customers c ON clo.customer_id = c.customer_id
WHERE DATEDIFF(CURRENT_DATE(), clo.last_order_date) > 90
  AND c.is_active = true
ORDER BY clo.total_spent DESC
LIMIT 20;

In [ ]:
%sql
-- ============================================
-- YOUR TURN: Challenge 5 -- Funnel Analysis
-- ============================================
-- Build a conversion funnel for order processing:
-- Step 1: Order placed (any status)
-- Step 2: Payment received (exists in payments table)
-- Step 3: Order shipped (status = 'shipped' or 'completed')
-- Step 4: Order completed (status = 'completed')
--
-- For each month, show:
-- total_orders, with_payment, shipped, completed,
-- payment_rate, ship_rate, completion_rate
--
-- Write your query below:


In [ ]:
%sql
-- SOLUTION: Funnel Analysis
SELECT
    date_format(o.order_date, 'yyyy-MM') AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT p.order_id) AS with_payment,
    COUNT(DISTINCT CASE WHEN o.status IN ('shipped', 'completed') THEN o.order_id END) AS shipped,
    COUNT(DISTINCT CASE WHEN o.status = 'completed' THEN o.order_id END) AS completed,
    -- Conversion rates
    ROUND(COUNT(DISTINCT p.order_id) * 100.0 / NULLIF(COUNT(DISTINCT o.order_id), 0), 1) AS payment_rate,
    ROUND(COUNT(DISTINCT CASE WHEN o.status IN ('shipped', 'completed') THEN o.order_id END) * 100.0 /
          NULLIF(COUNT(DISTINCT o.order_id), 0), 1) AS ship_rate,
    ROUND(COUNT(DISTINCT CASE WHEN o.status = 'completed' THEN o.order_id END) * 100.0 /
          NULLIF(COUNT(DISTINCT o.order_id), 0), 1) AS completion_rate
FROM orders o
LEFT JOIN payments p ON o.order_id = p.order_id AND p.payment_status = 'completed'
WHERE o.order_date >= '2023-01-01'
GROUP BY date_format(o.order_date, 'yyyy-MM')
ORDER BY month
LIMIT 20;

---
# Section 9: Recursive CTEs

## Querying Hierarchical Data

Recursive CTEs are essential for:
- Employee-manager hierarchies (org charts)
- Category trees (product taxonomy)
- Bill of materials (manufacturing)
- Graph traversal in SQL

In [ ]:
%sql
-- Recursive CTE: Employee hierarchy (org chart)
WITH RECURSIVE org_tree AS (
    -- Base case: top-level managers (no manager)
    SELECT
        employee_id,
        first_name,
        job_title,
        department,
        manager_id,
        0 AS level,
        CAST(first_name AS STRING) AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive case: employees reporting to someone in the tree
    SELECT
        e.employee_id,
        e.first_name,
        e.job_title,
        e.department,
        e.manager_id,
        t.level + 1,
        CONCAT(t.path, ' > ', e.first_name)
    FROM employees e
    JOIN org_tree t ON e.manager_id = t.employee_id
    WHERE t.level < 5  -- Prevent infinite recursion
)
SELECT
    REPEAT('  ', level) || first_name AS org_chart,
    job_title,
    department,
    level,
    path
FROM org_tree
ORDER BY path
LIMIT 30;

---
# Summary & Interview Preparation

## Advanced SQL Pattern Cheat Sheet

| Pattern | Use Case | Key Technique |
|---------|----------|---------------|
| **Cohort Analysis** | Retention, LTV by cohort | First event CTE + months_between |
| **Sessionization** | Detect activity sessions | LAG + gap detection + cumulative SUM |
| **Gap & Island** | Consecutive sequences | ROW_NUMBER subtraction trick |
| **Funnel** | Conversion tracking | Conditional COUNT + LEFT JOIN |
| **YoY Comparison** | Period-over-period | Self-join on year offset |
| **Running Top-N** | Top items per group | ROW_NUMBER + PARTITION BY |
| **MERGE/CDC** | Incremental updates | Conditional MERGE actions |
| **PIVOT/UNPIVOT** | Data reshaping | PIVOT clause or CASE WHEN |
| **Recursive CTE** | Hierarchies | WITH RECURSIVE + UNION ALL |
| **Window Frames** | Rolling calculations | ROWS BETWEEN ... AND ... |

## Practice Order (Easiest to Hardest)

1. Window functions with custom frames
2. ROLLUP / GROUPING SETS
3. Funnel analysis
4. Cohort retention
5. Sessionization
6. Gap and island (consecutive detection)
7. Recursive CTEs
8. Query optimization (EXPLAIN analysis)

---
*Advanced SQL for Data Engineering -- Using techmart_dw datasets*

In [ ]:
%sql
-- Final verification: all tables accessible
SELECT 'customers' AS tbl, COUNT(*) AS rows FROM customers
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'payments', COUNT(*) FROM payments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'inventory', COUNT(*) FROM inventory
UNION ALL SELECT 'shipments', COUNT(*) FROM shipments
ORDER BY tbl;